In [1]:
#################################################################
### Figs 5. Maps showing the Best RCM                         ###
#################################################################

import pandas as pd
import pygmt as pygmt

##############################################################
### Rank CORDEX (excluding ERA-I) for each grid based on 
### Median of Bias (absolute difference)

df = pd.read_csv('../Data/MedianBias_281grids.csv')

feature_cols = ['HIRHAM (EAS)', 'HadRM3P (EAS)',
                'RegCM4-1 (WAS)', 'RegCM4-4 (WAS)', 'HadRM3P (WAS)', 'RCA4 (WAS)', 'CCLM (EAS)']  ### preci & temp

ranked = df[feature_cols].copy()
ranked= abs(ranked-0)  ### whichever is closer to 0 is better

#add empty columns with column names
rank_no = ['Rk_1','Rk_2','Rk_3', 'Rk_4','Rk_5','Rk_6','Rk_7'] 
ranked[rank_no] = ""

for i in range(len(ranked)):
    ranked.loc[i, rank_no] = dict(sorted(dict(ranked[feature_cols].loc[i]).items(),reverse=False,key=lambda item:item[1])).keys()

ranked = df.join(ranked[rank_no] , how='left')

##############################################################
### This extracts the best (1st Rank) RCMs for each grid 
### measured by Median of Bias (Absolute Difference)
### for precipitation/temperature for daily/monthly/annually for all-season/JJASO

ranked = ranked[['Coord', 'Var', 'Season', 'TScale', 'Rk_1']]
# Set MultiIndex
ranked = ranked.set_index(['Coord', 'Var', 'Season', 'TScale'])

unstacked_df = ranked.unstack()
unstacked_df = unstacked_df.unstack()
unstacked_df = unstacked_df.unstack()
unstacked_df = unstacked_df.reset_index()
unstacked_df.columns = ['_'.join(map(str, x)) for x in unstacked_df.columns]
unstacked_df[['Lat', 'Lon']]= unstacked_df['Coord___'].str.split('_', expand=True)
unstacked_df[['Lat', 'Lon']] = unstacked_df[['Lat', 'Lon']].astype(float) 

#################################################################
### Fig 5A Precipitation
### This creates a map showing the best RCMs for precipitation 
### measured by Median of Bias (abolute difference)

df = unstacked_df
df = df.set_index(['Lat', 'Lon'])
df = df.filter(like='_preci', axis=1)

# mapping # to GCMs
mapping = {'CCLM (EAS)': 0,
           'HIRHAM (EAS)': 1, 
           'HadRM3P (EAS)': 2, 
           'RegCM4-1 (WAS)': 3,
           'RegCM4-4 (WAS)': 4,
           'HadRM3P (WAS)': 5,
           'RCA4 (WAS)': 6}
df = df.replace(mapping)

df = df.reset_index()  # set Lat & Lon as columns

cb_annots = ['CCLM (EAS)', 'HIRHAM (EAS)', 'HadRM3P (EAS)', 'RegCM4-1 (WAS)', 'RegCM4-4 (WAS)', 'HadRM3P (WAS)', 'RCA4 (WAS)' ]
cb_colors =  ["blueviolet", "red", "blue", "green", "sienna", "orange", "cyan"]  #magenta, blueviolet
Var = [df.Rk_1_Day_All_preci, df.Rk_1_Mon_All_preci, df.Rk_1_Yr_All_preci,
       df.Rk_1_Day_JJASO_preci, df.Rk_1_Mon_JJASO_preci, df.Rk_1_Yr_JJASO_preci]
labels = ['a) Day, All-S', 'b) Mon, All-S', 'c) Yr, All-S',
          'd) Day, JJASO', 'e) Mon, JJASO', 'f) Yr, JJASO']


fig = pygmt.Figure()

# Use pygmt.info to get region bounds (xmin, xmax, ymin, ymax)
# The below example will return a numpy array like [30.0, 60.0, 12.0, 22.0]
region = pygmt.info(
    data=df[["Lon", "Lat"]],  # x and y columns
    per_column=True,  # Report the min/max values per column as a numpy array
    # Round the min/max values of the first two columns to the nearest multiple of 3 and 2, respectively
    spacing=(3, 2)
)
# customize colormap
pygmt.makecpt(
    cmap=",".join(cb_colors),
    # 7 descrete colors
    series=[0, 6, 1],
    color_model="+c" + ",".join(cb_annots),
)

with pygmt.config(FONT_HEADING="18p", FONT_TITLE="18p,5", MAP_TITLE_OFFSET="1p", MAP_FRAME_TYPE="plain", FONT_ANNOT_PRIMARY="12p"):
    # Draw 6 subplots
    with fig.subplot(nrows=1, ncols=6, figsize=("48.14c", "12.33c"),# sum of all suplots
                    title="Best RCMs for Precipitation Measured by Median of Error", 
                    sharex="b", sharey="l", 
                    margins=["0.9c","1c"]):
        
        for j in range(6):
            fig.basemap(
                region=region, projection="R?", frame=["af", f"WSne+t{labels[j]}"], 
                panel=[0, j]
            )
            fig.plot(
                x=df.Lon,
                y=df.Lat,
                fill=Var[j].astype(dtype="category"),
                # Use colormap created by makecpt
                cmap=True,
                # Do not clip symbols that fall close to the plot bounds
                no_clip=True,
                # Use circles as symbols (the first "c") with diameter in
                # centimeters (the second "c")
                style="s0.3c",#"c0.1c",
                # Set transparency level for all symbols to deal with overplotting
                transparency=0
            )  
fig.colorbar(cmap=True, position="jBC+w48c/0.8c+o0c/-2.4c+h",equalsize=0.9,) 
fig.savefig("../Charts/Fig5A_preci_avg_MedianBias_1RCM_281grids.tif", dpi=300)
#fig.show()

#################################################################
### Fig 5B Temperature
### This creates a map showing the best RCMs for temperature 
### measured by Median of Bias (abolute difference)

df = unstacked_df
df = df.set_index(['Lat', 'Lon'])
df = df.filter(like='_temp', axis=1)

# mapping # to GCMs
mapping = {'CCLM (EAS)': 0,
           'HIRHAM (EAS)': 1, 
           'HadRM3P (EAS)': 2, 
           'RegCM4-1 (WAS)': 3,
           'RegCM4-4 (WAS)': 4,
           'HadRM3P (WAS)': 5,
           'RCA4 (WAS)': 6}
df = df.replace(mapping)

df = df.reset_index()  # set Lat & Lon as columns

cb_annots = ['CCLM (EAS)', 'HIRHAM (EAS)', 'HadRM3P (EAS)', 'RegCM4-1 (WAS)', 'RegCM4-4 (WAS)', 'HadRM3P (WAS)', 'RCA4 (WAS)' ]
cb_colors =  ["blueviolet", "red", "blue", "green", "sienna", "orange", "cyan"]  #magenta, blueviolet
Var = [df.Rk_1_Day_All_temp, df.Rk_1_Mon_All_temp, df.Rk_1_Yr_All_temp,
       df.Rk_1_Day_JJASO_temp, df.Rk_1_Mon_JJASO_temp, df.Rk_1_Yr_JJASO_temp]
labels = labels = ['a) Day, All-S', 'b) Mon, All-S', 'c) Yr, All-S',
          'd) Day, JJASO', 'e) Mon, JJASO', 'f) Yr, JJASO']


fig = pygmt.Figure()

# Use pygmt.info to get region bounds (xmin, xmax, ymin, ymax)
# The below example will return a numpy array like [30.0, 60.0, 12.0, 22.0]
region = pygmt.info(
    data=df[["Lon", "Lat"]],  # x and y columns
    per_column=True,  # Report the min/max values per column as a numpy array
    # Round the min/max values of the first two columns to the nearest multiple
    # of 3 and 2, respectively
    spacing=(3, 2)
)

pygmt.makecpt(
    cmap=",".join(cb_colors),
    # 7 descrete colors
    series=[0, 6, 1],
    color_model="+c" + ",".join(cb_annots),
)

with pygmt.config(FONT_HEADING="18p", FONT_TITLE="18p,5", MAP_TITLE_OFFSET="1p", MAP_FRAME_TYPE="plain", FONT_ANNOT_PRIMARY="12p"):
    # Draw 6 subplots
    with fig.subplot(nrows=1, ncols=6, figsize=("48.14c", "12.33c"),# sum of all suplots
                    title="Best RCMs for Temperature Measured by Median of Error", 
                    sharex="b", sharey="l",  
                    margins=["0.9c","1c"]):
        
        for j in range(6):
            fig.basemap(
                region=region, projection="R?", frame=["af", f"WSne+t{labels[j]}"], 
                panel=[0, j]
            )
            fig.plot(
                x=df.Lon,
                y=df.Lat,
                fill=Var[j].astype("category"),#.cat.codes.astype(int),
                # Use colormap created by makecpt
                cmap=True,
                # Do not clip symbols that fall close to the plot bounds
                no_clip=True,
                # Use circles as symbols (the first "c") with diameter in
                # centimeters (the second "c")
                style="s0.3c",#"c0.1c",
                # Set transparency level for all symbols to deal with overplotting
                transparency=0
            )  
fig.colorbar(cmap=True, position="jBC+w48c/0.8c+o0c/-2.4c+h",equalsize=0.9,) 
fig.savefig("../Charts/Fig5B_temp_avg_MedianBias_1RCM_281grids.tif", dpi=300)
#fig.show()
         
         

In [ ]:
#################################################################
### Figs 6. Maps showing the Best RCM                         ###
#################################################################

##############################################################
### Rank CORDEX (excluding ERA-I) for each grid 
### based on (95th quantile of Bias - 5th quantile of Bias) (absolute difference)

import pandas as pd
import pygmt as pygmt

df = pd.read_csv('../Data/q95-q05Bias_281grids.csv')

feature_cols = ['HIRHAM (EAS)', 'HadRM3P (EAS)',
                'RegCM4-1 (WAS)', 'RegCM4-4 (WAS)', 'HadRM3P (WAS)', 'RCA4 (WAS)', 'CCLM (EAS)']  ### preci & temp

ranked = df[feature_cols].copy()
ranked= abs(ranked-0)  ### whichever is closer to 0 is better

#add empty columns with column names
rank_no = ['Rk_1','Rk_2','Rk_3', 'Rk_4','Rk_5','Rk_6','Rk_7'] 
ranked[rank_no] = ""

for i in range(len(ranked)):
    ranked.loc[i, rank_no] = dict(sorted(dict(ranked[feature_cols].loc[i]).items(),reverse=False,key=lambda item:item[1])).keys()

ranked = df.join(ranked[rank_no] , how='left') 

##############################################################
### This extracts the best (1st Rank) RCMs for each grid 
### measured by the spread of Bias, that is, (95th quantile of Bias - 05th quantile of Bias) (Absolute Difference)
### for precipitation/temperature for daily/monthly/annually for all-season/JJASO

ranked = ranked[['Coord', 'Var', 'Season', 'TScale', 'Rk_1']]
# Set MultiIndex
ranked= ranked.set_index(['Coord', 'Var', 'Season', 'TScale'])

unstacked_df = ranked.unstack()
unstacked_df = unstacked_df.unstack()
unstacked_df = unstacked_df.unstack()
unstacked_df = unstacked_df.reset_index()
unstacked_df.columns = ['_'.join(map(str, x)) for x in unstacked_df.columns]
unstacked_df[['Lat', 'Lon']]= unstacked_df['Coord___'].str.split('_', expand=True)
unstacked_df[['Lat', 'Lon']] = unstacked_df[['Lat', 'Lon']].astype(float) 

#unstacked_df.to_csv('./data/q95-q05Bias_rank_RCMs_unstacked.csv', index=True, encoding='utf-8')

#################################################################
### Fig 6A Precipitation
### This creates a map showing the best RCMs for precipitation 
### measured by (95th - q05th) quantile of Bias (abolute difference)
df = unstacked_df
df = df.set_index(['Lat', 'Lon'])
df = df.filter(like='_preci', axis=1)

# mapping # to GCMs
mapping = {'CCLM (EAS)': 0,
           'HIRHAM (EAS)': 1, 
           'HadRM3P (EAS)': 2, 
           'RegCM4-1 (WAS)': 3,
           'RegCM4-4 (WAS)': 4,
           'HadRM3P (WAS)': 5,
           'RCA4 (WAS)': 6}
df = df.replace(mapping)

df = df.reset_index()  # set Lat & Lon as columns

cb_annots = ['CCLM (EAS)', 'HIRHAM (EAS)', 'HadRM3P (EAS)', 'RegCM4-1 (WAS)', 'RegCM4-4 (WAS)', 'HadRM3P (WAS)', 'RCA4 (WAS)' ]
cb_colors =  ["blueviolet", "red", "blue", "green", "sienna", "orange", "cyan"]  #magenta, blueviolet
Var = [df.Rk_1_Day_All_preci, df.Rk_1_Mon_All_preci, df.Rk_1_Yr_All_preci,
       df.Rk_1_Day_JJASO_preci, df.Rk_1_Mon_JJASO_preci, df.Rk_1_Yr_JJASO_preci]
labels = ['a) Day, All-S', 'b) Mon, All-S', 'c) Yr, All-S',
          'd) Day, JJASO', 'e) Mon, JJASO', 'f) Yr, JJASO']


fig = pygmt.Figure()

# Use pygmt.info to get region bounds (xmin, xmax, ymin, ymax)
# The below example will return a numpy array like [30.0, 60.0, 12.0, 22.0]
region = pygmt.info(
    data=df[["Lon", "Lat"]],  # x and y columns
    per_column=True,  # Report the min/max values per column as a numpy array
    # Round the min/max values of the first two columns to the nearest multiple
    # of 3 and 2, respectively
    spacing=(3, 2)
)

pygmt.makecpt(
    cmap=",".join(cb_colors),
    # 7 descrete colors
    series=[0, 6, 1],
    color_model="+c" + ",".join(cb_annots),
)

with pygmt.config(FONT_HEADING="18p", FONT_TITLE="18p,5", MAP_TITLE_OFFSET="1p", MAP_FRAME_TYPE="plain", FONT_ANNOT_PRIMARY="12p"):
    # Draw 6 subplots
    with fig.subplot(nrows=1, ncols=6, figsize=("48.14c", "12.33c"),# sum of all suplots
                    title="Best RCMs for Precipitation Measured by Spread of Error", 
                    sharex="b", sharey="l", 
                    margins=["0.9c","1c"]):
        
        for j in range(6):
            fig.basemap(
                region=region, projection="R?", frame=["af", f"WSne+t{labels[j]}"], 
                panel=[0, j]
            )
            fig.plot(
                x=df.Lon,
                y=df.Lat,
                fill=Var[j].astype("category"),
                # Use colormap created by makecpt
                cmap=True,
                # Do not clip symbols that fall close to the plot bounds
                no_clip=True,
                # Use circles as symbols (the first "c") with diameter in
                # centimeters (the second "c")
                style="s0.3c",#"c0.1c",
                # Set transparency level for all symbols to deal with overplotting
                transparency=0
            ) 

fig.colorbar(cmap=True, position="jBC+w48c/0.8c+o0c/-2.4c+h",equalsize=0.9,) 
fig.savefig("../Charts/Fig6A_preci_avg_q95-q05Bias_1RCM_281grids.tif", dpi=300)
#fig.show()  

#################################################################
### Fig 6B Temperature
### This creates a map showing the best RCMs for temperature 
### measured by (95th - 05th) quantile of Bias (abolute difference)

df = unstacked_df
df = df.set_index(['Lat', 'Lon'])
df = df.filter(like='_temp', axis=1)

# mapping # to GCMs
mapping = {'CCLM (EAS)': 0,
           'HIRHAM (EAS)': 1, 
           'HadRM3P (EAS)': 2, 
           'RegCM4-1 (WAS)': 3,
           'RegCM4-4 (WAS)': 4,
           'HadRM3P (WAS)': 5,
           'RCA4 (WAS)': 6}
df = df.replace(mapping)

df = df.reset_index()  # set Lat & Lon as columns

cb_annots = ['CCLM (EAS)', 'HIRHAM (EAS)', 'HadRM3P (EAS)', 'RegCM4-1 (WAS)', 'RegCM4-4 (WAS)', 'HadRM3P (WAS)', 'RCA4 (WAS)' ]
cb_colors =  ["blueviolet", "red", "blue", "green", "sienna", "orange", "cyan"]  #magenta, blueviolet
Var = [df.Rk_1_Day_All_temp, df.Rk_1_Mon_All_temp, df.Rk_1_Yr_All_temp,
       df.Rk_1_Day_JJASO_temp, df.Rk_1_Mon_JJASO_temp, df.Rk_1_Yr_JJASO_temp]
labels = ['a) Day, All-S', 'b) Mon, All-S', 'c) Yr, All-S',
          'd) Day, JJASO', 'e) Mon, JJASO', 'f) Yr, JJASO']


fig = pygmt.Figure()

# Use pygmt.info to get region bounds (xmin, xmax, ymin, ymax)
# The below example will return a numpy array like [30.0, 60.0, 12.0, 22.0]
region = pygmt.info(
    data=df[["Lon", "Lat"]],  # x and y columns
    per_column=True,  # Report the min/max values per column as a numpy array
    # Round the min/max values of the first two columns to the nearest multiple
    # of 3 and 2, respectively
    spacing=(3, 2),
)

pygmt.makecpt(
    cmap=",".join(cb_colors),
    # 7 descrete colors
    series=[0, 6, 1],
    color_model="+c" + ",".join(cb_annots),
)

with pygmt.config(FONT_HEADING="18p", FONT_TITLE="18p,5", MAP_TITLE_OFFSET="1p", MAP_FRAME_TYPE="plain", FONT_ANNOT_PRIMARY="12p"):
    # 6 subplots
    with fig.subplot(nrows=1, ncols=6, figsize=("48.14c", "12.33c"),# sum of all suplots
                    title="Best RCMs for Temperature Measured by Spread of Error", 
                    sharex="b", sharey="l", 
                    margins=["0.9c","1c"]):#
        
        for j in range(6):
            fig.basemap(
                region=region, projection="R?", frame=["af", f"WSne+t{labels[j]}"], 
                panel=[0, j]
            )
            fig.plot(
                x=df.Lon,
                y=df.Lat,
                fill=Var[j].astype("category"),#.cat.codes.astype(int),
                # Use colormap created by makecpt
                cmap=True,
                # Do not clip symbols that fall close to the plot bounds
                no_clip=True,
                # Use circles as symbols (the first "c") with diameter in
                # centimeters (the second "c")
                style="s0.3c",#"c0.1c",
                # Set transparency level for all symbols to deal with overplotting
                transparency=0
            ) 

fig.colorbar(cmap=True, position="jBC+w48c/0.8c+o0c/-2.4c+h",equalsize=0.9,) 
fig.savefig("../Charts/Fig6B_temp_avg_q95-q05Bias_1RCM_281grids.tif", dpi=300)
#fig.show()
              

In [ ]:
#################################################################
### Figs 7. Maps showing the Best RCM                         ###
#################################################################

##############################################################
### Rank CORDEX (excluding ERA-I) for each grid based on 
### Pearson correlation

import pandas as pd
import pygmt as pygmt

df = pd.read_csv('../Data/corr_281grids.csv',na_values='-')
df.fillna(-99,inplace=True)

feature_cols = ['HIRHAM (EAS)', 'HadRM3P (EAS)','RegCM4-1 (WAS)', 'RegCM4-4 (WAS)', 
                'HadRM3P (WAS)', 'RCA4 (WAS)', 'CCLM (EAS)']  ### preci & temp

ranked = df[feature_cols].copy()

#add empty columns with column names
rank_no = ['Rk_1','Rk_2','Rk_3', 'Rk_4','Rk_5','Rk_6','Rk_7'] 
ranked[rank_no] = ""

for i in range(len(ranked)):
    ranked.loc[i, rank_no] = dict(sorted(dict(ranked[feature_cols].loc[i]).items(),key=lambda item:str(item[1]),reverse=True)).keys()                                                                       

ranked = df.join(ranked[rank_no] , how='left')

##############################################################
### This extracts the best (1st Rank) RCMs for each grid 
### measured by Pearson Correlation Coefficient
### for precipitation/temperature for daily/monthly/annually for all-season/JJASO

ranked = ranked[['Coord', 'Var', 'Season', 'TScale', 'Rk_1']]
# Set MultiIndex
ranked = ranked.set_index(['Coord', 'Var', 'Season', 'TScale'])

unstacked_df = ranked.unstack()
unstacked_df = unstacked_df.unstack()
unstacked_df = unstacked_df.unstack()
unstacked_df = unstacked_df.reset_index()
unstacked_df.columns = ['_'.join(map(str, x)) for x in unstacked_df.columns]
unstacked_df[['Lat', 'Lon']]= unstacked_df['Coord___'].str.split('_', expand=True)
unstacked_df[['Lat', 'Lon']] = unstacked_df[['Lat', 'Lon']].astype(float) 

##############################################################
### Fig 7A Precipitation corr
### This creates a map showing the best RCMs for precipitation 
### measured by Pearson Correlation Coefficient

df = unstacked_df
df = df.set_index(['Lat', 'Lon'])
df = df.filter(like='_preci', axis=1)

# mapping # to GCMs
mapping = {'CCLM (EAS)': 0,
           'HIRHAM (EAS)': 1, 
           'HadRM3P (EAS)': 2, 
           'RegCM4-1 (WAS)': 3,
           'RegCM4-4 (WAS)': 4,
           'HadRM3P (WAS)': 5,
           'RCA4 (WAS)': 6}
df = df.replace(mapping)

df = df.reset_index()  # set Lat & Lon as columns

cb_annots = ['CCLM (EAS)', 'HIRHAM (EAS)', 'HadRM3P (EAS)', 'RegCM4-1 (WAS)', 'RegCM4-4 (WAS)', 'HadRM3P (WAS)', 'RCA4 (WAS)' ]
cb_colors =  ["blueviolet", "red", "blue", "green", "sienna", "orange", "cyan"]  
Var = [df.Rk_1_Day_All_preci, df.Rk_1_Mon_All_preci, df.Rk_1_Yr_All_preci,
       df.Rk_1_Day_JJASO_preci, df.Rk_1_Mon_JJASO_preci, df.Rk_1_Yr_JJASO_preci]
labels = ['a) Day, All-S', 'b) Mon, All-S', 'c) Yr, All-S',
          'd) Day, JJASO', 'e) Mon, JJASO', 'f) Yr, JJASO']


fig = pygmt.Figure()

# Use pygmt.info to get region bounds (xmin, xmax, ymin, ymax)
# The below example will return a numpy array like [30.0, 60.0, 12.0, 22.0]
region = pygmt.info(
    data=df[["Lon", "Lat"]],  # x and y columns
    per_column=True,  # Report the min/max values per column as a numpy array
    # Round the min/max values of the first two columns to the nearest multiple
    # of 3 and 2, respectively
    spacing=(3, 2)
)

pygmt.makecpt(
    cmap=",".join(cb_colors),
    # 7 descrete colors
    series=[0, 6, 1],
    color_model="+c" + ",".join(cb_annots),
)

with pygmt.config(FONT_HEADING="18p", FONT_TITLE="18p,5", MAP_TITLE_OFFSET="1p", MAP_FRAME_TYPE="plain", FONT_ANNOT_PRIMARY="12p"): 
    # 6 subplots
    with fig.subplot(nrows=1, ncols=6, figsize=("48.14c", "12.33c"),# sum of all suplots
                    title="Best RCMs for Precipitation Measured by Pearson's r", 
                    sharex="b", sharey="l", 
                    margins=["0.9c","1c"]):## horizontal, vertical, left, right margins
        
        for j in range(6):
            fig.basemap(
                region=region, projection="R?", frame=["af", f"WSne+t{labels[j]}"], 
                panel=[0, j]
            )
            fig.plot(
                x=df.Lon,
                y=df.Lat,
                fill=Var[j].astype("category"),
                # Use colormap created by makecpt
                cmap=True,
                # Do not clip symbols that fall close to the plot bounds
                no_clip=True,
                # Use circles as symbols (the first "c") with diameter in
                # centimeters (the second "c")
                style="s0.3c",#"c0.1c",
                # Set transparency level for all symbols to deal with overplotting
                transparency=0
            )  

fig.colorbar(cmap=True, position="jBC+w48c/0.8c+o0c/-2.4c+h", equalsize=0.9,) 
fig.savefig("../Charts/Fig7A_preci_avg_corr_1RCM_281grids.tif", dpi=300)

##############################################################
### Fig 7B Temperature corr
### This creates a map showing the best RCMs for temperature 
### measured by Pearson correlation

df = unstacked_df
df = df.set_index(['Lat', 'Lon'])
df = df.filter(like='_temp', axis=1)

# mapping # to GCMs
mapping = {'CCLM (EAS)': 0,
           'HIRHAM (EAS)': 1, 
           'HadRM3P (EAS)': 2, 
           'RegCM4-1 (WAS)': 3,
           'RegCM4-4 (WAS)': 4,
           'HadRM3P (WAS)': 5,
           'RCA4 (WAS)': 6}
df = df.replace(mapping)

df = df.reset_index()  # set Lat & Lon as columns

cb_annots = ['CCLM (EAS)', 'HIRHAM (EAS)', 'HadRM3P (EAS)', 'RegCM4-1 (WAS)', 'RegCM4-4 (WAS)', 'HadRM3P (WAS)', 'RCA4 (WAS)' ]
cb_colors =  ["blueviolet", "red", "blue", "green", "sienna", "orange", "cyan"]  
Var = [df.Rk_1_Day_All_temp, df.Rk_1_Mon_All_temp, df.Rk_1_Yr_All_temp,
       df.Rk_1_Day_JJASO_temp, df.Rk_1_Mon_JJASO_temp, df.Rk_1_Yr_JJASO_temp]
labels = ['a) Day, All-S', 'b) Mon, All-S', 'c) Yr, All-S',
          'd) Day, JJASO', 'e) Mon, JJASO', 'f) Yr, JJASO']


fig = pygmt.Figure()

# Use pygmt.info to get region bounds (xmin, xmax, ymin, ymax)
# The below example will return a numpy array like [30.0, 60.0, 12.0, 22.0]
region = pygmt.info(
    data=df[["Lon", "Lat"]],  # x and y columns
    per_column=True,  # Report the min/max values per column as a numpy array
    # Round the min/max values of the first two columns to the nearest multiple
    # of 3 and 2, respectively
    spacing=(3, 2)
)

pygmt.makecpt(
    cmap=",".join(cb_colors),
    # 7 descrete colors
    series=[0, 6, 1],
    color_model="+c" + ",".join(cb_annots),
)

with pygmt.config(FONT_HEADING="18p", FONT_TITLE="18p,5", MAP_TITLE_OFFSET="1p", MAP_FRAME_TYPE="plain", FONT_ANNOT_PRIMARY="12p"): 
    # 6 subplots
    with fig.subplot(nrows=1, ncols=6, figsize=("48.14c", "12.33c"),# sum of all suplots
                    title="Best RCMs for Temperature Measured by Pearson's r", 
                    sharex="b", sharey="l", 
                    margins=["0.9c","1c"]):
        
        for j in range(6):
            fig.basemap(
                region=region, projection="R?", frame=["af", f"WSne+t{labels[j]}"], 
                panel=[0, j]
            )
            fig.plot(
                x=df.Lon,
                y=df.Lat,
                fill=Var[j].astype("category"),
                # Use colormap created by makecpt
                cmap=True,
                # Do not clip symbols that fall close to the plot bounds
                no_clip=True,
                # Use circles as symbols (the first "c") with diameter in
                # centimeters (the second "c")
                style="s0.3c",#"c0.1c",
                # Set transparency level for all symbols to deal with overplotting
                transparency=0
            )  

fig.colorbar(cmap=True, position="jBC+w48c/0.8c+o0c/-2.4c+h", equalsize=0.9,) 
fig.savefig("../Charts/Fig7B_temp_avg_corr_1RCM_281grids.tif", dpi=300)
        
         

In [ ]:
#################################################################
### Figs 8. Maps showing the Best RCM                         ###
#################################################################

import pandas as pd
import pygmt as pygmt

##############################################################
### Rank CORDEX (excluding ERA-I) for each grid based on RMSE

df = pd.read_csv('../Data/rmse_281grids.csv')

feature_cols = ['HIRHAM (EAS)', 'HadRM3P (EAS)',
                'RegCM4-1 (WAS)', 'RegCM4-4 (WAS)', 'HadRM3P (WAS)', 'RCA4 (WAS)', 'CCLM (EAS)']  ### preci & temp

ranked = df[feature_cols].copy()

#add empty columns with column names
rank_no = ['Rk_1','Rk_2','Rk_3', 'Rk_4','Rk_5','Rk_6','Rk_7'] 
ranked[rank_no] = ""

for i in range(len(ranked)):
    ranked.loc[i, rank_no] = dict(sorted(dict(ranked[feature_cols].loc[i]).items(),reverse=False,key=lambda item:item[1])).keys()

ranked = df.join(ranked[rank_no] , how='left')

##############################################################
### This extracts the best (1st Rank) RCMs for each grid 
### measured by RMSE
### for precipitation/temperature for daily/monthly/annually for all-season/JJASO

ranked = ranked[['Coord', 'Var', 'Season', 'TScale', 'Rk_1']]
# Set MultiIndex
ranked = ranked.set_index(['Coord', 'Var', 'Season', 'TScale'])

unstacked_df = ranked.unstack()
unstacked_df = unstacked_df.unstack()
unstacked_df = unstacked_df.unstack()
unstacked_df = unstacked_df.reset_index()
unstacked_df.columns = ['_'.join(map(str, x)) for x in unstacked_df.columns]
unstacked_df[['Lat', 'Lon']]= unstacked_df['Coord___'].str.split('_', expand=True)
unstacked_df[['Lat', 'Lon']] = unstacked_df[['Lat', 'Lon']].astype(float) 


##############################################################
### Fig 8A Precipitation rmse
### This creates a map showing the best RCMs for precipitation 
### measured by RMSE

df = unstacked_df
df = df.set_index(['Lat', 'Lon'])
df = df.filter(like='_preci', axis=1)

# mapping # to GCMs
mapping = {'CCLM (EAS)': 0,
           'HIRHAM (EAS)': 1, 
           'HadRM3P (EAS)': 2, 
           'RegCM4-1 (WAS)': 3,
           'RegCM4-4 (WAS)': 4,
           'HadRM3P (WAS)': 5,
           'RCA4 (WAS)': 6}
df = df.replace(mapping)

df = df.reset_index()  # set Lat & Lon as columns

cb_annots = ['CCLM (EAS)', 'HIRHAM (EAS)', 'HadRM3P (EAS)', 'RegCM4-1 (WAS)', 'RegCM4-4 (WAS)', 'HadRM3P (WAS)', 'RCA4 (WAS)' ]
cb_colors =  ["blueviolet", "red", "blue", "green", "sienna", "orange", "cyan"]  
Var = [df.Rk_1_Day_All_preci, df.Rk_1_Mon_All_preci, df.Rk_1_Yr_All_preci,
       df.Rk_1_Day_JJASO_preci, df.Rk_1_Mon_JJASO_preci, df.Rk_1_Yr_JJASO_preci]
labels = ['a) Day, All-S', 'b) Mon, All-S', 'c) Yr, All-S',
          'd) Day, JJASO', 'e) Mon, JJASO', 'f) Yr, JJASO']


fig = pygmt.Figure()

# Use pygmt.info to get region bounds (xmin, xmax, ymin, ymax)
# The below example will return a numpy array like [30.0, 60.0, 12.0, 22.0]
region = pygmt.info(
    data=df[["Lon", "Lat"]],  # x and y columns
    per_column=True,  # Report the min/max values per column as a numpy array
    # Round the min/max values of the first two columns to the nearest multiple
    # of 3 and 2, respectively
    spacing=(3, 2)
)

pygmt.makecpt(
    cmap=",".join(cb_colors),
    # 7 descrete colors
    series=[0, 6, 1],
    color_model="+c" + ",".join(cb_annots),
)

with pygmt.config(FONT_HEADING="18p", FONT_TITLE="18p,5", MAP_TITLE_OFFSET="1p", MAP_FRAME_TYPE="plain", FONT_ANNOT_PRIMARY="12p"): 
    # 6 subplots
    with fig.subplot(nrows=1, ncols=6, figsize=("48.14c", "12.33c"),# sum of all suplots
                    title="Best RCMs for Precipitation Measured by RMSE", 
                    sharex="b", sharey="l", 
                    margins=["0.9c","1c"]):
        
        for j in range(6):
            fig.basemap(
                region=region, projection="R?", frame=["af", f"WSne+t{labels[j]}"], 
                panel=[0, j]
            )
            fig.plot(
                x=df.Lon,
                y=df.Lat,
                fill=Var[j].astype("category"),
                # Use colormap created by makecpt
                cmap=True,
                # Do not clip symbols that fall close to the plot bounds
                no_clip=True,
                # Use circles as symbols (the first "c") with diameter in
                # centimeters (the second "c")
                style="s0.3c",#"c0.1c",
                # Set transparency level for all symbols to deal with overplotting
                transparency=0
            ) 

fig.colorbar(cmap=True, position="jBC+w48c/0.8c+o0c/-2.4c+h",equalsize=0.9,) #FONT_ANNOT_PRIMARY
fig.savefig("../Charts/Fig8A_preci_avg_rmse_1RCM_281grids.tif", dpi=300)

##############################################################
### Fig 8B Temperature rmse
### This creates a map showing the best RCMs for temperature 
### measured by RMSE

df = unstacked_df
df = df.set_index(['Lat', 'Lon'])
df = df.filter(like='_temp', axis=1)

# mapping # to GCMs
mapping = {'CCLM (EAS)': 0,
           'HIRHAM (EAS)': 1, 
           'HadRM3P (EAS)': 2, 
           'RegCM4-1 (WAS)': 3,
           'RegCM4-4 (WAS)': 4,
           'HadRM3P (WAS)': 5,
           'RCA4 (WAS)': 6}
df = df.replace(mapping)

df = df.reset_index()  # set Lat & Lon as columns

cb_annots = ['CCLM (EAS)', 'HIRHAM (EAS)', 'HadRM3P (EAS)', 'RegCM4-1 (WAS)', 'RegCM4-4 (WAS)', 'HadRM3P (WAS)', 'RCA4 (WAS)' ]
cb_colors =  ["blueviolet", "red", "blue", "green", "sienna", "orange", "cyan"] 
Var = [df.Rk_1_Day_All_temp, df.Rk_1_Mon_All_temp, df.Rk_1_Yr_All_temp,
       df.Rk_1_Day_JJASO_temp, df.Rk_1_Mon_JJASO_temp, df.Rk_1_Yr_JJASO_temp]
labels = ['a) Day, All-S', 'b) Mon, All-S', 'c) Yr, All-S',
          'd) Day, JJASO', 'e) Mon, JJASO', 'f) Yr, JJASO']

fig = pygmt.Figure()

# Use pygmt.info to get region bounds (xmin, xmax, ymin, ymax)
# The below example will return a numpy array like [30.0, 60.0, 12.0, 22.0]
region = pygmt.info(
    data=df[["Lon", "Lat"]],  # x and y columns
    per_column=True,  # Report the min/max values per column as a numpy array
    # Round the min/max values of the first two columns to the nearest multiple
    # of 3 and 2, respectively
    spacing=(3, 2)
)

pygmt.makecpt(
    cmap=",".join(cb_colors),
    # 7 descrete colors
    series=[0, 6, 1],
    color_model="+c" + ",".join(cb_annots),
)

with pygmt.config(FONT_HEADING="18p", FONT_TITLE="18p,5", MAP_TITLE_OFFSET="1p", MAP_FRAME_TYPE="plain", FONT_ANNOT_PRIMARY="12p"):
    # 6 subplots
    with fig.subplot(nrows=1, ncols=6, figsize=("48.14c", "12.33c"),# sum of all suplots
                    title="Best RCMs for Temperature Measured by RMSE", 
                    sharex="b", sharey="l", 
                    margins=["0.9c","1c"]):
        
        for j in range(6):
            fig.basemap(
                region=region, projection="R?", frame=["af", f"WSne+t{labels[j]}"], 
                panel=[0, j]
            )
            fig.plot(
                x=df.Lon,
                y=df.Lat,
                fill=Var[j].astype("category"),
                # Use colormap created by makecpt
                cmap=True,
                # Do not clip symbols that fall close to the plot bounds
                no_clip=True,
                # Use circles as symbols (the first "c") with diameter in
                # centimeters (the second "c")
                style="s0.3c",#"c0.1c",
                # Set transparency level for all symbols to deal with overplotting
                transparency=0
            )  

fig.colorbar(cmap=True, position="jBC+w48c/0.8c+o0c/-2.4c+h",equalsize=0.9,) #FONT_ANNOT_PRIMARY 
fig.savefig("../Charts/Fig8B_temp_avg_rmse_1RCM_281grids.tif", dpi=300)
                  

In [ ]:
#################################################################
### Figs 9. Maps showing the Best RCM                         ###
#################################################################

import pandas as pd
import pygmt as pygmt

##############################################################
### Rank CORDEX (excluding ERA-I) for each grid based on CRMS

df = pd.read_csv('../Data/crms_281grids.csv')

feature_cols = ['HIRHAM (EAS)', 'HadRM3P (EAS)',
                'RegCM4-1 (WAS)', 'RegCM4-4 (WAS)', 'HadRM3P (WAS)', 'RCA4 (WAS)', 'CCLM (EAS)']  ### preci & temp

ranked = df[feature_cols].copy()

#add empty columns with column names
rank_no = ['Rk_1','Rk_2','Rk_3', 'Rk_4','Rk_5','Rk_6','Rk_7'] 
ranked[rank_no] = ""

for i in range(len(ranked)):
    ranked.loc[i, rank_no] = dict(sorted(dict(ranked[feature_cols].loc[i]).items(),reverse=False,key=lambda item:item[1])).keys()

ranked = df.join(ranked[rank_no] , how='left')

##############################################################
### This extracts the best (1st Rank) RCMs for each grid 
### measured by CRMS
### for precipitation/temperature for daily/monthly/annually for all-season/JJASO

ranked = ranked[['Coord', 'Var', 'Season', 'TScale', 'Rk_1']]
# Set MultiIndex
ranked = ranked.set_index(['Coord', 'Var', 'Season', 'TScale'])

unstacked_df = ranked.unstack()
unstacked_df = unstacked_df.unstack()
unstacked_df = unstacked_df.unstack()
unstacked_df = unstacked_df.reset_index()
unstacked_df.columns = ['_'.join(map(str, x)) for x in unstacked_df.columns]
unstacked_df[['Lat', 'Lon']]= unstacked_df['Coord___'].str.split('_', expand=True)
unstacked_df[['Lat', 'Lon']] = unstacked_df[['Lat', 'Lon']].astype(float) 

##############################################################
### Fig 9A Precipitation crms
### This creates a map showing the best RCMs for precipitation 
### measured by CRMS

df = unstacked_df
df = df.set_index(['Lat', 'Lon'])
df = df.filter(like='_preci', axis=1)

# mapping # to GCMs
mapping = {'CCLM (EAS)': 0,
           'HIRHAM (EAS)': 1, 
           'HadRM3P (EAS)': 2, 
           'RegCM4-1 (WAS)': 3,
           'RegCM4-4 (WAS)': 4,
           'HadRM3P (WAS)': 5,
           'RCA4 (WAS)': 6}
df = df.replace(mapping)

df = df.reset_index()  # set Lat & Lon as columns

cb_annots = ['CCLM (EAS)', 'HIRHAM (EAS)', 'HadRM3P (EAS)', 'RegCM4-1 (WAS)', 'RegCM4-4 (WAS)', 'HadRM3P (WAS)', 'RCA4 (WAS)' ]
cb_colors =  ["blueviolet", "red", "blue", "green", "sienna", "orange", "cyan"]  
Var = [df.Rk_1_Day_All_preci, df.Rk_1_Mon_All_preci, df.Rk_1_Yr_All_preci,
       df.Rk_1_Day_JJASO_preci, df.Rk_1_Mon_JJASO_preci, df.Rk_1_Yr_JJASO_preci]
labels = ['a) Day, All-S', 'b) Mon, All-S', 'c) Yr, All-S',
          'd) Day, JJASO', 'e) Mon, JJASO', 'f) Yr, JJASO']


fig = pygmt.Figure()

# Use pygmt.info to get region bounds (xmin, xmax, ymin, ymax)
# The below example will return a numpy array like [30.0, 60.0, 12.0, 22.0]
region = pygmt.info(
    data=df[["Lon", "Lat"]],  # x and y columns
    per_column=True,  # Report the min/max values per column as a numpy array
    # Round the min/max values of the first two columns to the nearest multiple
    # of 3 and 2, respectively
    spacing=(3, 2)
)

pygmt.makecpt(
    cmap=",".join(cb_colors),
    # 7 descrete colors
    series=[0, 6, 1],
    color_model="+c" + ",".join(cb_annots),
)

with pygmt.config(FONT_HEADING="18p", FONT_TITLE="18p,5", MAP_TITLE_OFFSET="1p", MAP_FRAME_TYPE="plain", FONT_ANNOT_PRIMARY="12p"): 
    # 6 subplots
    with fig.subplot(nrows=1, ncols=6, figsize=("48.14c", "12.33c"),# sum of all suplots
                    title="Best RCMs for Precipitation Measured by CRMSE", 
                    sharex="b", sharey="l", 
                    margins=["0.9c","1c"]):
        
        for j in range(6):
            fig.basemap(
                region=region, projection="R?", frame=["af", f"WSne+t{labels[j]}"], 
                panel=[0, j]
            )
            fig.plot(
                x=df.Lon,
                y=df.Lat,
                fill=Var[j].astype("category"),
                # Use colormap created by makecpt
                cmap=True,
                # Do not clip symbols that fall close to the plot bounds
                no_clip=True,
                # Use circles as symbols (the first "c") with diameter in
                # centimeters (the second "c")
                style="s0.3c",#"c0.1c",
                # Set transparency level for all symbols to deal with overplotting
                transparency=0
            ) 

fig.colorbar(cmap=True, position="jBC+w48c/0.8c+o0c/-2.4c+h", equalsize=0.9,) 
fig.savefig("../Charts/Fig9A_preci_avg_crms_1RCM_281grids.tif", dpi=300)

##############################################################
### Fig 9B Temperature crms
### This creates a map showing the best RCMs for temperature 
### measured by CRMS

df = unstacked_df
df = df.set_index(['Lat', 'Lon'])
df = df.filter(like='_temp', axis=1)

# mapping # to GCMs
mapping = {'CCLM (EAS)': 0,
           'HIRHAM (EAS)': 1, 
           'HadRM3P (EAS)': 2, 
           'RegCM4-1 (WAS)': 3,
           'RegCM4-4 (WAS)': 4,
           'HadRM3P (WAS)': 5,
           'RCA4 (WAS)': 6}
df = df.replace(mapping)

df = df.reset_index()  # set Lat & Lon as columns

cb_annots = ['CCLM (EAS)', 'HIRHAM (EAS)', 'HadRM3P (EAS)', 'RegCM4-1 (WAS)', 'RegCM4-4 (WAS)', 'HadRM3P (WAS)', 'RCA4 (WAS)' ]
cb_colors =  ["blueviolet", "red", "blue", "green", "sienna", "orange", "cyan"]  
Var = [df.Rk_1_Day_All_temp, df.Rk_1_Mon_All_temp, df.Rk_1_Yr_All_temp,
       df.Rk_1_Day_JJASO_temp, df.Rk_1_Mon_JJASO_temp, df.Rk_1_Yr_JJASO_temp]
labels = ['a) Day, All-S', 'b) Mon, All-S', 'c) Yr, All-S',
          'd) Day, JJASO', 'e) Mon, JJASO', 'f) Yr, JJASO']


fig = pygmt.Figure()

# Use pygmt.info to get region bounds (xmin, xmax, ymin, ymax)
# The below example will return a numpy array like [30.0, 60.0, 12.0, 22.0]
region = pygmt.info(
    data=df[["Lon", "Lat"]],  # x and y columns
    per_column=True,  # Report the min/max values per column as a numpy array
    # Round the min/max values of the first two columns to the nearest multiple
    # of 3 and 2, respectively
    spacing=(3, 2)
)

pygmt.makecpt(
    cmap=",".join(cb_colors),
    # 7 descrete colors
    series=[0, 6, 1],
    color_model="+c" + ",".join(cb_annots),
)

with pygmt.config(FONT_HEADING="18p", FONT_TITLE="18p,5", MAP_TITLE_OFFSET="1p", MAP_FRAME_TYPE="plain", FONT_ANNOT_PRIMARY="12p"): 
    # 6 subplots
    with fig.subplot(nrows=1, ncols=6, figsize=("48.14c", "12.33c"),# sum of all suplots
                    title="Best RCMs for Temperature Measured by CRMSE", 
                    sharex="b", sharey="l", 
                    margins=["0.9c","1c"]):
        
        for j in range(6):
            fig.basemap(
                region=region, projection="R?", frame=["af", f"WSne+t{labels[j]}"], 
                panel=[0, j]
            )
            fig.plot(
                x=df.Lon,
                y=df.Lat,
                fill=Var[j].astype("category"),
                # Use colormap created by makecpt
                cmap=True,
                # Do not clip symbols that fall close to the plot bounds
                no_clip=True,
                # Use circles as symbols (the first "c") with diameter in
                # centimeters (the second "c")
                style="s0.3c",#"c0.1c",
                # Set transparency level for all symbols to deal with overplotting
                transparency=0
            )  

fig.colorbar(cmap=True, position="jBC+w48c/0.8c+o0c/-2.4c+h",equalsize=0.9,) #FONT_ANNOT_PRIMARY
fig.savefig("../Charts/Fig9B_temp_avg_crms_1RCM_281grids.tif", dpi=300)
                  